# Value of personalization: a two-point population treated as two types or as a continuum

The population consists of two narrow clusters of delay sensitivities around the
two-type values of the load experiment. It is analyzed in two ways:

* **Two-type treatment**: each cluster is one type that receives one depth and one
  priority position (the 86-policy optimization of the two-type experiments).
* **Continuum treatment**: each of the $n$ equal-mass types receives its own depth,
  with nonpreemptive $c\mu$ priority (the monotone optimization of the continuum
  experiment).

For each we report the centralized optimum, the best incentive-compatible menu,
and the welfare gap. Exact stationary M/G/1 formulas throughout.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sgd_calibration import load_calibration

cal = load_calibration(proxy="cost")
VALUES, MEANS, SECONDS = cal["values"], cal["means"], cal["seconds"]
EFFORTS = list(cal["efforts"])
print("values ", np.round(VALUES, 3))
print("means  ", np.round(MEANS, 4))

## Population

Cluster centres and shares follow the two-type load experiment: $\theta_1=0.12$ for
the impatient $75\%$ and $\theta_2=0.09$ for the patient $25\%$. Each cluster is
spread uniformly over a narrow band of half-width `HALF_WIDTH`.

In [ ]:
THETA1, SHARE1 = 0.12, 0.75          # impatient cluster
THETA2, SHARE2 = 0.09, 0.25          # patient cluster
HALF_WIDTH = 0.002
N = 40
N2, N1 = int(round(N * SHARE2)), int(round(N * SHARE1))
THETA = np.r_[np.linspace(THETA2 - HALF_WIDTH, THETA2 + HALF_WIDTH, N2),
              np.linspace(THETA1 - HALF_WIDTH, THETA1 + HALF_WIDTH, N1)]   # ascending: patient cluster first
QUANTILE = (np.arange(N) + 0.5) / N
LAMBDAS = np.round(np.arange(0.30, 1.2001, 0.02), 4)
LAMBDA_STAR = 0.90
OUTPUT = Path("results") / "continuous_type_personalization"
M_EXT = np.r_[MEANS, 0.0]; B_EXT = np.r_[SECONDS, 0.0]; V_EXT = np.r_[VALUES, 0.0]   # index -1 -> d_0

## Two-type treatment (86 policies)

In [ ]:
RULES_BOTH = ("1NP", "2NP", "FCFS")
POLICIES2 = [(d1, d2, rule) for d1, d2 in itertools.product(range(-1, 5), repeat=2)
             for rule in (RULES_BOTH if (d1 >= 0 and d2 >= 0) else ("FCFS",))]
assert len(POLICIES2) == 86


def sojourn2(d1, d2, lam1, lam2, rule):
    adm = (d1 >= 0, d2 >= 0)
    lam = (lam1 * adm[0], lam2 * adm[1])
    m = (MEANS[d1] if adm[0] else 0.0, MEANS[d2] if adm[1] else 0.0)
    b = (SECONDS[d1] if adm[0] else 0.0, SECONDS[d2] if adm[1] else 0.0)
    rho = (lam[0] * m[0], lam[1] * m[1]); total = rho[0] + rho[1]
    if total >= 1.0:
        return None
    R = 0.5 * (lam[0] * b[0] + lam[1] * b[1]); w = [0.0, 0.0]
    if rule == "FCFS" or not (adm[0] and adm[1]):
        for i in range(2):
            if adm[i]:
                w[i] = m[i] + R / (1.0 - total)
        return tuple(w)
    h, l = (0, 1) if rule[0] == "1" else (1, 0)
    w[h] = m[h] + R / (1.0 - rho[h]); w[l] = m[l] + R / ((1.0 - rho[h]) * (1.0 - total))
    return tuple(w)


def two_type(lam):
    lam1, lam2 = lam * SHARE1, lam * SHARE2
    central = ic = None
    for d1, d2, rule in POLICIES2:
        w = sojourn2(d1, d2, lam1, lam2, rule)
        if w is None:
            continue
        W = (lam1 * (VALUES[d1] - THETA1 * w[0]) if d1 >= 0 else 0.0) + (lam2 * (VALUES[d2] - THETA2 * w[1]) if d2 >= 0 else 0.0)
        rec = dict(d1=d1, d2=d2, rule=rule, welfare=W, w1=w[0], w2=w[1])
        if central is None or W > central["welfare"]:
            central = rec
        if w[0] <= w[1] + 1e-12 and (ic is None or W > ic["welfare"]):
            ic = rec
    return central, ic

## Continuum treatment (monotone depth assignments, nonpreemptive $c\mu$ priority)

In [ ]:
def evaluate(depths, lam, rule="NP"):
    depths = np.asarray(depths); served = depths >= 0; li = lam / N
    m, b, v = M_EXT[depths], B_EXT[depths], V_EXT[depths]
    rho = li * m; total = rho.sum()
    if total >= 1.0:
        return -np.inf, None
    R = 0.5 * li * b.sum(); w = np.zeros(N)
    if rule == "FCFS":
        w[served] = m[served] + R / (1.0 - total)
    else:
        idx = np.flatnonzero(served)
        order = idx[np.argsort(-(THETA[idx] / m[idx]), kind="stable")]
        sig = np.cumsum(rho[order]); sig_prev = sig - rho[order]
        w[order] = m[order] + R / ((1.0 - sig_prev) * (1.0 - sig))
    return (li * (v - THETA * w))[served].sum(), w


def evaluate_monotone_batch(counts, lam):
    cum = np.cumsum(counts, axis=1)[:, :-1]
    level = (np.arange(N)[None, :] >= cum[:, :, None]).sum(axis=1)
    depth = 4 - level
    m, b, v = M_EXT[depth], B_EXT[depth], V_EXT[depth]
    li = lam / N; rho = li * m; total = rho.sum(axis=1); R = 0.5 * li * b.sum(axis=1)
    served = depth >= 0
    sig = np.cumsum(rho[:, ::-1], axis=1)[:, ::-1]; sig_prev = sig - rho
    w = np.where(served, m + R[:, None] / ((1.0 - sig_prev) * (1.0 - sig)), 0.0)
    W = (li * (v - THETA[None, :] * w) * served).sum(axis=1)
    return np.where(total < 1.0, W, -np.inf)


def compositions(n, parts):
    out = []
    for bars in itertools.combinations(range(n + parts - 1), parts - 1):
        prev, row = -1, []
        for bb in bars:
            row.append(bb - prev - 1); prev = bb
        row.append(n + parts - 2 - prev); out.append(row)
    return np.array(out, dtype=np.int16)


COUNTS = compositions(N, 6)


def continuum(lam, chunk=200_000):
    best_W, best_c = -np.inf, None
    for s in range(0, len(COUNTS), chunk):
        W = evaluate_monotone_batch(COUNTS[s:s + chunk], lam); j = int(np.argmax(W))
        if W[j] > best_W:
            best_W, best_c = float(W[j]), COUNTS[s + j]
    d = np.repeat(np.arange(4, -2, -1), best_c)
    # unrestricted local search from the monotone optimum
    Wc, improved = best_W, True
    while improved:
        improved = False
        for i in range(N):
            cur = d[i]
            for k in range(-1, 5):
                if k == cur:
                    continue
                d[i] = k; Wk, _ = evaluate(d, lam)
                if Wk > Wc + 1e-12:
                    Wc, cur, improved = Wk, k, True
            d[i] = cur
    W, w = evaluate(d, lam)
    ic = bool(np.all(np.diff(w[d >= 0]) <= 1e-12))           # sojourn non-increasing in theta among served
    return dict(welfare=W, depths=d, w=w, ic=ic, nonmonotone_gain=Wc - best_W)

## Sweep over the arrival rate

In [ ]:
rows = []
for lam in LAMBDAS:
    c2, i2 = two_type(lam); cc = continuum(lam)
    rows.append(dict(lam=lam, W2c=max(c2["welfare"], 0.0), W2ic=max(i2["welfare"], 0.0), Wc=max(cc["welfare"], 0.0),
                     ic=cc["ic"], gain=cc["nonmonotone_gain"], two=c2, two_ic=i2, cont=cc))
W2c = np.array([r["W2c"] for r in rows]); W2ic = np.array([r["W2ic"] for r in rows]); Wc = np.array([r["Wc"] for r in rows])
print("continuum optimum IC at every load:", all(r["ic"] for r in rows),
      "| largest non-monotone gain:", f"{max(r['gain'] for r in rows):.1e}")
print("continuum optimum >= two-type optimum at every load:", bool(np.all(Wc >= W2c - 1e-12)))
lab = lambda d: "d0" if d < 0 else EFFORTS[d]
print("\n lam   two-type central     two-type IC        continuum   | gap(two-type)  continuum gain over two-type central")
for r in rows[::5]:
    c2 = r["two"]
    print(f"{r['lam']:.2f}   {r['W2c']:.4f} ({lab(c2['d1'])}/{lab(c2['d2'])},{c2['rule']})   {r['W2ic']:.4f}   {r['Wc']:.4f}   |  "
          f"{100*(r['W2c']-r['W2ic'])/max(r['W2c'],1e-12):5.1f}%       {100*(r['Wc']-r['W2c'])/max(r['W2c'],1e-12):5.1f}%")

## Figure 1: welfare

In [ ]:
RED = "#c0392b"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "axes.linewidth": 0.6, "lines.linewidth": 1.5})
fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(LAMBDAS, Wc, color="#222222", linewidth=2.4, label="Continuum")
ax.plot(LAMBDAS, W2c, color="#1f77b4", linewidth=2.0, linestyle="-.", label="Two-type: centralized")
ax.plot(LAMBDAS, W2ic, color=RED, linewidth=2.0, linestyle="--", label="Two-type: IC menu")
ax.set_xlim(LAMBDAS[0], LAMBDAS[-1])
ax.set_xlabel(r"Arrival rate $\lambda$", fontsize=24)
ax.set_ylabel(r"$\mathcal{W}$", fontsize=26, rotation=0, labelpad=18)
ax.yaxis.set_label_coords(-0.2, 0.5)
ax.tick_params(labelsize=18)
ax.legend(fontsize=14, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.4, borderpad=0.4, columnspacing=1.0)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.21, right=0.96, top=0.82, bottom=0.21)
fig.savefig(OUTPUT.with_name("continuous_type_personalization_welfare.png"), dpi=300)
fig.savefig(OUTPUT.with_name("continuous_type_personalization_welfare.pdf"))
plt.show()

## Figure 2: service depth assignment at $\lambda^\star$

The horizontal axis is the customer's position in the population, ordered by
delay sensitivity: the first $25\%$ form the patient cluster and the rest the
impatient cluster.

In [ ]:
i_star = int(np.argmin(np.abs(LAMBDAS - LAMBDA_STAR))); r = rows[i_star]
c2, i2, cc = r["two"], r["two_ic"], r["cont"]
series = {
    "two":  (np.r_[np.full(N2, c2["d2"]), np.full(N1, c2["d1"])], np.r_[np.full(N2, c2["w2"]), np.full(N1, c2["w1"])],
             dict(color="#1f77b4", linewidth=2.0, linestyle="-.", label="Two-type: centralized")),
    "ic":   (np.r_[np.full(N2, i2["d2"]), np.full(N1, i2["d1"])], np.r_[np.full(N2, i2["w2"]), np.full(N1, i2["w1"])],
             dict(color=RED, linewidth=2.0, linestyle="--", label="Two-type: IC menu")),
    "cont": (cc["depths"], cc["w"], dict(color="#222222", linewidth=2.4, linestyle="-", label="Continuum")),
}
print(f"lambda*={r['lam']}: two-type optimum {lab(c2['d1'])}/{lab(c2['d2'])} {c2['rule']} W={c2['welfare']:.4f}; "
      f"two-type IC menu {lab(i2['d1'])}/{lab(i2['d2'])} {i2['rule']} W={i2['welfare']:.4f}; "
      f"continuum W={cc['welfare']:.4f}, IC={cc['ic']}")
print("continuum depths (patient cluster | impatient cluster):",
      " ".join(lab(d) for d in cc["depths"][:N2]), "|", " ".join(lab(d) for d in cc["depths"][N2:]))


def cluster_shading(ax, y=0.04, va="bottom"):
    ax.axvspan(0, SHARE2, color="#f0f0f0", linewidth=0)
    ax.text(SHARE2 / 2, y, "patient\ncluster", ha="center", va=va, fontsize=13, transform=ax.get_xaxis_transform())
    ax.text(SHARE2 + 0.38 * SHARE1, y, "impatient cluster", ha="center", va=va, fontsize=13, transform=ax.get_xaxis_transform())


LEGEND = dict(fontsize=12, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=True,
              edgecolor="#cccccc", framealpha=1.0, handlelength=1.3, borderpad=0.4, columnspacing=0.8)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
cluster_shading(ax, y=0.5, va="center")
for key in ("two", "ic", "cont"):
    d, w, st = series[key]
    ax.step(QUANTILE, d + 1, where="mid", **st)
ax.set_xlim(0, 1); ax.set_ylim(-0.4, 5.5)
ax.set_yticks(range(6), [f"$d_{k}$" for k in range(6)])
ax.set_xlabel("Customer quantile by delay sensitivity", fontsize=20)
ax.tick_params(labelsize=18)
ax.legend(**LEGEND)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.13, right=0.96, top=0.87, bottom=0.21)
fig.savefig(OUTPUT.with_name("continuous_type_personalization_depth.png"), dpi=300)
fig.savefig(OUTPUT.with_name("continuous_type_personalization_depth.pdf"))
plt.show()